In [1]:
import glob
from safetensors.torch import load_file, load
import io, os

In [2]:
model_file = "./layoutlmv3-finetune/model.safetensors"
model_dir = "./layoutlmv3-finetune"

In [3]:
def split_file(input_file, chunk_size_mb=80):
    chunk_size = chunk_size_mb * 1024 * 1024
    with open(input_file, "rb") as f:
        part_num = 1
        while chunk := f.read(chunk_size):
            with open(f"{input_file}.part{part_num}", "wb") as pf:
                pf.write(chunk)
            part_num += 1


# Example
split_file(model_file, 80)

In [4]:
def load_split_safetensors(model_path: str):
    prefix = os.path.abspath(model_path)
    parts = sorted(glob.glob(f"{prefix}.part*"))
    if not parts:
        raise FileNotFoundError(f"No split parts found for {prefix}")

    buffer = io.BytesIO()
    for part in parts:
        with open(part, "rb") as f:
            buffer.write(f.read())
    buffer.seek(0)
    return load(buffer.getvalue())

In [5]:
from transformers import AutoProcessor, AutoModelForTokenClassification, AutoConfig

/Users/naveen1.mathur/Desktop/x/sftc/_learning/ml/mlp/ml-projects/_env_05_OCR_DOCUMENT_PARSER/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
processor = AutoProcessor.from_pretrained(model_dir, apply_ocr=False)
# model = AutoModelForTokenClassification.from_pretrained(model_dir, state_dict={})

In [7]:
config = AutoConfig.from_pretrained(model_dir)
model = AutoModelForTokenClassification.from_config(config)

In [8]:
weights = load_split_safetensors(os.path.join(model_dir, "model.safetensors"))

In [9]:
model.load_state_dict(weights, strict=True)

<All keys matched successfully>

In [10]:
model = model.to("mps")
model.eval()

LayoutLMv3ForTokenClassification(
  (layoutlmv3): LayoutLMv3Model(
    (embeddings): LayoutLMv3TextEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (x_position_embeddings): Embedding(1024, 128)
      (y_position_embeddings): Embedding(1024, 128)
      (h_position_embeddings): Embedding(1024, 128)
      (w_position_embeddings): Embedding(1024, 128)
    )
    (patch_embed): LayoutLMv3PatchEmbeddings(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
    (encoder): LayoutLMv3Encoder